# Count pairs

This notebook goes with `vak`. Each section takes one idea at its smallest size, small enough to check by hand.
The matching command then runs the same functions on 50,000 stories.

Before this: step 3 turns text into token ids. After this: step 5 writes text with these counts.

| Section | Idea | Used in |
|---|---|---|
| 1 | Count pairs: the smallest language model, and its loss | `vak counts` |

## 1. Count pairs: the smallest language model

A **language model** gives a chance to every token that could come next. The smallest one counts, in the training
stories, which token follows which. The chance of b after a is the count of the pair (a, b), divided by the count of
a. **Add-one smoothing** first adds 1 to every count, so no token gets a chance of 0. The toy has six tokens, and one
story to learn from: "the cat sat on the mat", with the end token before and after it.

In [1]:
from vak.counts import PairCounts

words = ["<|endoftext|>", "the", "cat", "sat", "on", "mat"]  # a token's id is its place in this list
story = []
for word in ["<|endoftext|>", "the", "cat", "sat", "on", "the", "mat", "<|endoftext|>"]:
    story.append(words.index(word))
print("the story as ids:", story)

toy = PairCounts.from_stories([story], vocab_size=len(words))
print('after "the":')
for word, count, chance in zip(words, toy.counts[1].tolist(), toy.chances[1].tolist()):
    print(f"  {word:15} count {count:.0f}   chance {chance:.3f}")

the story as ids: [0, 1, 2, 3, 4, 1, 5, 0]
after "the":
  <|endoftext|>   count 0   chance 0.125
  the             count 0   chance 0.125
  cat             count 1   chance 0.250
  sat             count 0   chance 0.125
  on              count 0   chance 0.125
  mat             count 1   chance 0.250


**What you see**

In training, "the" is followed once by cat and once by mat. Add-one gives each of the six counts one more, so
the row adds up to 2 + 6 = 8. Cat and mat get 2/8 = `0.250`. The four tokens never seen after "the" get
1/8 = `0.125` each. The six chances add up to 1.

The **loss** of one token is minus the log of the chance the model gave the true token. A sure, right guess costs
almost 0; a rare one costs a lot. The **held-out loss** is the mean over every token of stories the model never
trained on. Here the held-out story is "the cat sat on the cat".

In [2]:
import math

held_out = [0, 1, 2, 3, 4, 1, 2, 0]  # <|endoftext|> the cat sat on the cat <|endoftext|>
losses = toy.losses(held_out)
print("loss of each next token:", [round(loss, 3) for loss in losses])
print("held-out loss:", round(sum(losses) / len(losses), 3))
print("an even guess over 6 tokens:", round(math.log(6), 3))

loss of each next token: [1.253, 1.386, 1.253, 1.253, 1.253, 1.386, 1.946]
held-out loss: 1.39
an even guess over 6 tokens: 1.792


**What you see**

The first token, "the" after the end token, had chance (1 + 1) / (1 + 6) = 2/7, so its loss is `1.253`.
"cat" after "the" had chance 2/8, so it costs `1.386`. The last pair, "cat" then the end token, never appeared in
training. Smoothing still gives it 1/7, so it costs `1.946` and not infinity. The held-out loss is `1.39`, below
the `1.792` of an even guess: counting pairs helps.

Used in: `uv run vak counts` counts 12.6 million pairs in 50,000 stories and reports the loss on 2,000 held-out
stories. That number is the baseline the GPT must beat.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Count pairs | chance of b after a = (count of a, b + 1) / (count of a + vocabulary size) | the baseline every model must beat |
| Held-out loss | minus the log of the true token's chance, averaged over unseen stories; lower is better | the GPT reports through the same `held_out_loss`; the baseline scores 3.706 |